In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

Libraries imported successfully.


Uploading Dataset

In [8]:
from google.colab import files
uploaded = files.upload()

Saving loan_approval_dataset.csv to loan_approval_dataset (2).csv


Loading and Cleaning Data

In [17]:
df = pd.read_csv("loan_approval_dataset.csv")


df.columns = df.columns.str.strip()

print("Data Loaded. Shape:", df.shape)
df.head()

Data Loaded. Shape: (4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


Splitting the Data

In [18]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"].str.strip()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (3415, 12)
X_test shape: (854, 12)


Feature Engineering (Training Set)


In [19]:
label_encoders = {}
scaler = MinMaxScaler()
target_encoder = LabelEncoder()

y_train = target_encoder.fit_transform(y_train)

cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(exclude=['object']).columns


for col in X_train.columns:
    if col in cat_cols:
        X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        label_encoders[col] = le
    else:
        X_train[col] = X_train[col].fillna(X_train[col].median())

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

print("Training data preprocessing complete.")

Training data preprocessing complete.


Feature Engineering (Test Set)

In [20]:

y_test = target_encoder.transform(y_test)

for col in X_test.columns:
    if col in cat_cols:
        X_test[col] = X_test[col].fillna(X_train[col].mode()[0])
        X_test[col] = label_encoders[col].transform(X_test[col])
    else:
        X_test[col] = X_test[col].fillna(X_train[col].median())

X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Testing data preprocessing complete.")

Testing data preprocessing complete.


Model Training and Evaluation

In [21]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Training Complete.")
print(f"Model Accuracy = {accuracy * 100:.2f}%")

Model Training Complete.
Model Accuracy = 88.52%


In [22]:
import datetime
import random
import pandas as pd
import numpy as np


if 'X' not in locals():
    print(" ERROR: Training data not found in memory.")
    print("Please run your Model Training cells first!")
else:
    print("\n" + "="*60)
    print(f"{'INSTITUTIONAL CREDIT RISK ASSESSMENT TERMINAL':^60}")
    print("="*60)

    try:
        user_input_data = {
            'no_of_dependents': int(input("Number of Dependents: ")),
            'education': input("Educational Qualification (Graduate / Not Graduate): "),
            'self_employed': input("Employment Status - Self Employed (Yes / No): "),
            'income_annum': float(input("Annual Gross Income (INR): ")),
            'loan_amount': float(input("Principal Loan Amount Requested (INR): ")),
            'loan_term': int(input("Amortization Period (Total Months): ")),
            'cibil_score': int(input("CIBIL / Credit Bureau Score (300-900): ")),
            'residential_assets_value': float(input("Valuation - Residential Real Estate: ")),
            'commercial_assets_value': float(input("Valuation - Commercial Real Estate: ")),
            'luxury_assets_value': float(input("Valuation - High-Value Movable Assets: ")),
            'bank_asset_value': float(input("Liquid Asset Reserves (Bank Balance): "))
        }
    except ValueError:
        print("\n DATA ENTRY ERROR: Please ensure all financial figures are numeric.")
        user_input_data = None

    if user_input_data:

        live_df = pd.DataFrame([user_input_data])

        live_df = live_df.reindex(columns=X.columns, fill_value=0)

        for col in cat_cols:
            le = label_encoders[col]
            raw_val = str(live_df[col].iloc[0]).strip().title()
            possible_matches = [cls for cls in le.classes_ if raw_val in cls]
            if possible_matches:
                live_df[col] = le.transform([possible_matches[0]])
            else:
                live_df[col] = le.transform([le.classes_[0]])

        live_df[num_cols] = scaler.transform(live_df[num_cols])


        pred_numeric = knn.predict(live_df)
        pred_text = target_encoder.inverse_transform(pred_numeric)[0]
        prob = knn.predict_proba(live_df)[0]
        confidence = max(prob) * 100


        ref_id = "REF-" + str(random.randint(100000, 999999))
        timestamp = datetime.datetime.now().strftime("%d-%b-%Y | %H:%M:%S")
        final_status = str(pred_text).strip().upper()

        print("\n" + "="*60)
        print(f"{'OFFICIAL CREDIT EVALUATION SUMMARY':^60}")
        print("="*60)
        print(f" Transaction Reference : {ref_id}")
        print(f" Timestamp             : {timestamp}")
        print(f" Credit Bureau Score   : {user_input_data['cibil_score']}")
        print(f" Debt-to-Income Ratio  : {(user_input_data['loan_amount']/user_input_data['income_annum']):.2f}")
        print("-" * 60)

        if 'APPROVED' in final_status:
            print(f" DECISION STATUS      :  PROVISIONALLY APPROVED")
            print(f" SYSTEM CONFIDENCE    : {confidence:.2f}%")
            print("-" * 60)
            print(" EVALUATION REMARKS   : Financial profile meets underwriting")
            print("                        guidelines. KYC verification required.")
        else:
            print(f" DECISION STATUS      :  REJECTED")
            print(f" SYSTEM CONFIDENCE    : {confidence:.2f}%")
            print("-" * 60)
            print(" EVALUATION REMARKS
            5  : Application exceeds risk thresholds.")
            print("                        Insufficient credit bureau standing.")

        print("="*60)
        print(f"{'END OF STATUTORY REPORT':^60}")
        print("="*60)


       INSTITUTIONAL CREDIT RISK ASSESSMENT TERMINAL        
Number of Dependents: 1
Educational Qualification (Graduate / Not Graduate): 5
Employment Status - Self Employed (Yes / No): 1
Annual Gross Income (INR): 5
Principal Loan Amount Requested (INR): 4
Amortization Period (Total Months): 5
CIBIL / Credit Bureau Score (300-900): 4
Valuation - Residential Real Estate: 5
Valuation - Commercial Real Estate: 4
Valuation - High-Value Movable Assets: 5
Liquid Asset Reserves (Bank Balance): 4

             OFFICIAL CREDIT EVALUATION SUMMARY             
 Transaction Reference : REF-464517
 Timestamp             : 28-Mar-2026 | 07:12:36
 Credit Bureau Score   : 4
 Debt-to-Income Ratio  : 0.80
------------------------------------------------------------
 DECISION STATUS      :  REJECTED
 SYSTEM CONFIDENCE    : 80.00%
------------------------------------------------------------
 EVALUATION REMARKS   : Application exceeds risk thresholds.
                        Insufficient credit bureau st

Saving the Model

In [24]:
model_data = {
    "model": knn,
    "scaler": scaler,
    "encoders": label_encoders,
    "target_encoder": target_encoder
}

with open("knn_loan_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Model saved as 'knn_loan_model.pkl'")

Model saved as 'knn_loan_model.pkl'
